# EasyRAG Building A Tiny Eval Set

## Chapter position

This chapter closes the loop. It separates retrieval quality from answer quality and shows how to turn runtime behavior into measurements you can actually debug.

## Learning question

What makes a tiny eval set useful instead of arbitrary?

## Success criteria

- build a small manual eval set with `EvalCase`
- inspect which fields belong to retrieval evaluation versus answer evaluation
- run the same case set through both evaluators

## Source code anchors

- `easyrag.rag.types.EvalCase`
- `easyrag.rag.evaluation.evaluate_retrieval`
- `easyrag.rag.evaluation.evaluate_answers`


## Method principles

- `easyrag.rag.types.EvalCase`: This dataclass defines one evaluation expectation. It ties a question to expected documents, snippets, abstention behavior, and optional reference text so evaluation stays deterministic.
- `easyrag.rag.evaluation.evaluate_retrieval`: This package-level entry point runs the retrieval evaluator over a case set and returns both case-level reports and aggregate metrics.
- `easyrag.rag.evaluation.evaluate_answers`: This package-level entry point runs grounded answer evaluation and aggregates citation presence, support ratio, and abstention behavior.

Design reason: these anchors keep runtime behavior and scoring logic close together. The notebook uses them so that a metric report still points back to concrete cases, retrieved evidence, and explicit expectations instead of becoming a detached benchmark number.


## How the code fits together

The flow in this notebook is questions and expectations -> eval cases -> retrieval and answer checks. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

Design reason: the notebook keeps case construction, per-case evidence, and aggregate metrics in one visible chain. That pacing prevents evaluation from becoming a black-box score generator and makes regression reports easier to audit.


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.evaluation import evaluate_answers, evaluate_retrieval

## What this cell is proving

The goal is not to build a big benchmark. The goal is to make a tiny case set that is explicit enough to diagnose behavior.

Design reason: the notebook turns evaluation into explicit cases, flags, and reports here so every score remains tied to visible evidence. That is why the later metrics can be challenged, recomputed, or debugged case by case.


In [ ]:
case_set = [
    EvalCase(
        question="What uses query rewrite?",
        expected_document_ids=("doc::retrieval",),
        expected_snippets=("query rewrite",),
        reference_answer="EasyRAG uses query rewrite for grounded retrieval.",
        expected_to_abstain=False,
        metadata={"split": "dev", "theme": "retrieval"},
    ),
    EvalCase(
        question="When is the company retreat?",
        expected_document_ids=(),
        expected_snippets=(),
        reference_answer="",
        expected_to_abstain=True,
        metadata={"split": "dev", "theme": "abstain"},
    ),
]
for case in case_set:
    _print_json(case)

## Why this output looks like this

The next cell runs the case set through both evaluators so you can see how the same cases support multiple kinds of scoring.

Design reason: the report is decomposed into simple flags and metrics so the score stays auditable. The next cell reuses the same case-level evidence rather than inventing a second scoring path, which is why the aggregate number can be sanity-checked by hand.


In [ ]:
case_tmp = tempfile.TemporaryDirectory()
case_rag = EasyRAG(
    working_dir=case_tmp.name,
    workspace="tiny-eval-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(case_rag.initialize_storages())
run_sync(
    case_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses query rewrite for grounded retrieval.\n",
            "# Policy\nCitation-aware answers preserve traceability.\n",
        ],
        ids=["doc::retrieval", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/policy.md"],
    )
)
retrieval_report = run_sync(
    evaluate_retrieval(
        case_rag,
        case_set,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)
answer_report = run_sync(
    evaluate_answers(
        case_rag,
        case_set,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
        AnswerParam(),
    )
)
_print_json(
    {
        "retrieval_metrics": retrieval_report["metrics"],
        "answer_metrics": answer_report["metrics"],
    }
)

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

The optional path simply checks that the same `EvalCase` objects can be reused with a provider-backed workspace when available.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-tiny-eval-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                "# Retrieval\nGrounded retrieval keeps answers inspectable.\n",
                ids=["doc::retrieval"],
                file_paths=["docs/retrieval.md"],
            )
        )
        provider_report = run_sync(
            evaluate_retrieval(
                provider_rag, [case_set[0]], QueryParam(mode="hybrid", chunk_top_k=2)
            )
        )
        _print_json(provider_report["metrics"])
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- A single metric rarely tells you which stage broke. Keep retrieval and answer quality separate.
- Eval cases should be small enough to audit by hand. If the expectation is vague, the metric will be vague too.
- Use metrics to narrow the search space, then inspect the case-level reports and runtime metadata.

## Takeaway

A tiny eval set becomes useful once each case is explicit about what success means. `expected_document_ids` and `expected_snippets` help retrieval scoring. `reference_answer` and `expected_to_abstain` help answer evaluation. The same case object can support several layers of diagnosis.

In [ ]:
run_sync(case_rag.finalize_storages())
case_tmp.cleanup()
print("Cleaned up the tiny-eval workspace.")

## Where to go next

- Continue with [06_03_retrieval_metrics.ipynb](06_03_retrieval_metrics.ipynb) to score retrieval quality in more detail.
- Read [principles/21-evaluation-and-debugging.md](../../docs/principles/21-evaluation-and-debugging.md) for the evaluation mindset behind small case sets.
- Revisit [06_01_evaluation_basics.ipynb](06_01_evaluation_basics.ipynb) if you want the stage-level overview again.